# Exploratory Data Analysis

Goals: understand target distribution, class imbalance, missingness, feature correlations, and systematically identify data leakage columns before any modeling begins.

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.load_data import LEAKAGE_COLUMNS, load_accepted_loans

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 1. Load data

In [ ]:
df = load_accepted_loans(nrows=200_000, drop_leakage=False)  # leakage cols visible for inspection
print(df.shape)
df[:10]

In [ ]:
df.columns.tolist()

## 2. Basic structure

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.describe(include='all').T[:50]

## 3. Target variable: `loan_status`

Lending Club has multiple statuses (Fully Paid, Current, Charged Off, Late (31-120 days), Late (16-30 days), In Grace Period, Default)

In [ ]:
df['loan_status'].value_counts(dropna=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df['loan_status'].value_counts().plot(kind='barh', ax=ax)
ax.set_title('Loan status distribution')
plt.tight_layout()
plt.show()

In [ ]:
TARGET_MAP = {
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
}

df_labeled = df[df['loan_status'].isin(TARGET_MAP.keys())].copy()
df_labeled['target'] = df_labeled['loan_status'].map(TARGET_MAP)

print(f"Labeled subset: {len(df_labeled):,} of {len(df):,} rows")
print(df_labeled['target'].value_counts(normalize=True))

## 4. Missingness analysis

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
missing_pct.head(30)

In [ ]:
# Columns with very high missingness are candidates for dropping
high_missing = missing_pct[missing_pct > 50]
print(f"{len(high_missing)} columns with >50% missing:")
high_missing

## 5. Systematic leakage column check

In [ ]:
print("Currently excluded by default in load_data.py:")
for col in LEAKAGE_COLUMNS:
    print(f"  - {col}")

all_cols = df.columns.tolist()
print(f"\nTotal columns in raw file: {len(all_cols)}")

## 6. Correlation / multicollinearity

In [ ]:
numeric_df = df_labeled.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.6})
ax.set_title('Numeric feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Pairs with |correlation| > 0.8 — candidates for dropping one of the pair
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[(corr_pairs < 1.0) & (corr_pairs > 0.8)]
corr_pairs.drop_duplicates()

In [ ]:
# Correlation analysis excluding leakage columns
from src.data.load_data import LEAKAGE_COLUMNS

numeric_df_clean = df_labeled.select_dtypes(include=[np.number]).drop(
    columns=[c for c in LEAKAGE_COLUMNS if c in numeric_df.columns]
)
corr_clean = numeric_df_clean.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_clean, cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.6})
ax.set_title('Numeric feature correlation matrix with leakage columns excluded')
plt.tight_layout()
plt.show()

print(f"Columns in full heatmap: {numeric_df.shape[1]}")
print(f"Columns in leakage-free heatmap: {numeric_df_clean.shape[1]}")

In [ ]:
corr_pairs_clean = corr_clean.abs().unstack().sort_values(ascending=False)
corr_pairs_clean = corr_pairs_clean[(corr_pairs_clean < 1.0) & (corr_pairs_clean > 0.8)]
corr_pairs_clean.drop_duplicates()

## 7. Target correlation with candidate features

In [ ]:
target_corr = numeric_df.corrwith(df_labeled['target']).abs().sort_values(ascending=False)
target_corr.head(20)